# 02 — Survey inference and predictive modeling

This review notebook inspects survey-weighted estimates, multiply imputed income results, and held-out predictive-model performance generated by the project pipeline.

Data Source: National Center for Health Statistics, National Health Interview Survey, 2025.

Analyses, interpretations, and conclusions are those of the author and are not attributable to NCHS.

In [1]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
PROCESSED = ROOT / 'data' / 'processed'

In [2]:
overall = pd.read_csv(PROCESSED / 'survey_overall.csv')
overall[['weighted_prevalence','standard_error','ci_lower','ci_upper','design_df']].round(4)

,weighted_prevalence,standard_error,ci_lower,ci_upper,design_df
0,0.0705,0.0022,0.0662,0.0748,562


## Multiply imputed income

All 10 official income imputations are analyzed separately and pooled.

![Pooled income estimates](../assets/income_prevalence.svg)

In [3]:
income = pd.read_csv(PROCESSED / 'income_pooled_estimates.csv')
income[['poverty_band','weighted_prevalence','ci_lower','ci_upper','fraction_missing_information','mi_df']].round(4)

,poverty_band,weighted_prevalence,ci_lower,ci_upper,fraction_missing_information,mi_df
0,Below 100% FPL,0.1038,0.0862,0.1214,0.1185,278.7999
1,100-199% FPL,0.1107,0.0977,0.1237,0.0691,408.4470
2,200-399% FPL,0.0860,0.0779,0.0942,0.0236,528.8990
3,400%+ FPL,0.0358,0.0314,0.0401,0.0155,543.3045


## Held-out prediction

![Held-out model discrimination](../assets/model_discrimination_metrics.svg)

In [4]:
metrics = pd.read_csv(PROCESSED / 'model_metrics.csv')
metrics[['model','weighted_roc_auc','weighted_average_precision','weighted_brier_score','weighted_log_loss']].round(4)

,model,weighted_roc_auc,weighted_average_precision,weighted_brier_score,weighted_log_loss
0,prevalence_baseline,0.5000,0.0720,0.0668,0.2587
1,core_logistic,0.7382,0.2306,0.0612,0.2296
2,expanded_logistic,0.7729,0.2719,0.0599,0.2221
3,expanded_gradient_boosting,0.7807,0.2667,0.0599,0.2205


In [5]:
income_models = pd.read_csv(PROCESSED / 'income_model_summary.csv')
income_models[['model','imputations','weighted_roc_auc_mean','weighted_roc_auc_sd','weighted_average_precision_mean']].round(4)

,model,imputations,weighted_roc_auc_mean,weighted_roc_auc_sd,weighted_average_precision_mean
0,core_plus_income_logistic,10,0.7513,0.0015,0.2311
1,expanded_plus_income_logistic,10,0.7812,0.0010,0.2711


### Interpretation

Expanded logistic regression improves discrimination over the access/demographic core. Gradient boosting provides a modest ROC-AUC gain but does not dominate average precision. Income adds a small, stable improvement across all 10 imputations.